In [21]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
torch.set_default_device(device)
print(device.type + " set")

cuda set


In [22]:
from torch.utils.data import TensorDataset, DataLoader

def read_idx3_images(path):
    with open(path, 'rb') as f:
        # Skip the 16-byte header: magic number(4), count(4), rows(4), cols(4)
        data = np.fromfile(f, dtype=np.uint8, offset=16)
    # Reshape to (Number of images, 784) to match your neural network input
    return data.reshape(-1, 784).astype(np.float32) / 255.0

def read_idx1_labels(path):
    with open(path, 'rb') as f:
        # Skip the 8-byte header: magic number(4), count(4)
        data = np.fromfile(f, dtype=np.uint8, offset=8)
    return data.astype(np.int64)

train_images = read_idx3_images('./micrograd/code/data/train-images.idx3-ubyte')
train_labels = read_idx1_labels('./micrograd/code/data/train-labels.idx1-ubyte')

x_train = torch.tensor(train_images, dtype=torch.float32)
y_train = torch.tensor(train_labels, dtype=torch.long)

train_ds = TensorDataset(x_train, y_train)
train_loader = DataLoader(train_ds, batch_size=32, shuffle=True, generator=torch.Generator(device=device))


In [23]:
model = nn.Sequential(
    nn.Linear(784, 16),
    nn.GELU(),
    nn.Linear(16, 16),
    nn.GELU(),
    nn.Linear(16, 10)
)

dummy_input = torch.randn(1, 784)
dummy_output = torch.randn(1, 10)

# feed forward.
output = model(dummy_input)

print(f"Output shape: {output.shape}") # [1, 10]
print(output)

Output shape: torch.Size([1, 10])
tensor([[-0.2319, -0.2005, -0.1272, -0.1869, -0.0334,  0.0450, -0.2168, -0.1309,
         -0.0043, -0.0386]], device='cuda:0', grad_fn=<AddmmBackward0>)


In [24]:
optimizer = optim.Adam(model.parameters(), lr=0.001)

# training loop
for epoch in range(10):

    for images, labels in train_loader:
        optimizer.zero_grad()
        
        outputs = model(images)
        loss = nn.CrossEntropyLoss()(outputs, labels)

        loss.backward()
        optimizer.step()
    print(f"Epoch {epoch+1} complete. Final batch loss: {loss.item():.4f}")

Epoch 1 complete. Final batch loss: 0.1470
Epoch 2 complete. Final batch loss: 0.1826
Epoch 3 complete. Final batch loss: 0.0607
Epoch 4 complete. Final batch loss: 0.1244
Epoch 5 complete. Final batch loss: 0.1522
Epoch 6 complete. Final batch loss: 0.0128
Epoch 7 complete. Final batch loss: 0.1520
Epoch 8 complete. Final batch loss: 0.4384
Epoch 9 complete. Final batch loss: 0.0577
Epoch 10 complete. Final batch loss: 0.0791


In [25]:
output = nn.Softmax(model(dummy_input))
output

Softmax(
  dim=tensor([[  8.3136, -10.7429,  -0.5040,  -0.6318, -10.0828,  -8.8092, -11.3972,
             3.4794,   2.2362,  -0.9636]], device='cuda:0',
         grad_fn=<AddmmBackward0>)
)